# F1 Strategy Engine
### End-to-end ML pipeline for race strategy prediction — 5 models, zero leakage

---

| # | Model | Target | Algorithm | Key Score |
|---|-------|--------|-----------|-----------|
| M1 | Lap-Time Regressor | `LapTimeSec` | LightGBM + Optuna | Test RMSE **4.298** · MAE **1.971** |
| M2 | Pit-Lap Classifier | `IsPitLap` | CatBoost + Optuna | ROC-AUC **0.8226** · PR-AUC **0.1439** |
| M3 | Pit-in-3 Classifier | `WillPitWithin3Laps` | CatBoost + Optuna | ROC-AUC **0.7856** · F1 **0.3308** |
| M4 | Long-Horizon Regressor | `LapsUntilNextPit` (all) | CatBoost + Optuna | Test RMSE **7.677** · MAE **5.503** |
| M5 | Short-Horizon Regressor | `LapsUntilNextPit` (≤10) | CatBoost + Optuna | Test RMSE **2.786** · MAE **2.292** |

> **Data split:** `Year ≤ 2022` → train · `2023` → validation · `2024` → held-out test (strictly temporal, no leakage)


---
## 0 · Environment Setup

Install all dependencies once so every subsequent cell runs cleanly.

In [ ]:
%pip install -q optuna catboost lightgbm shap joblib


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    precision_recall_curve, PrecisionRecallDisplay,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder

from lightgbm import LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor

pd.set_option("display.max_columns", None)
print("All packages loaded ✓")


---
## 1 · Shared Utilities

All helper functions defined once here and reused across every model section.

In [ ]:
# ── Data splitting ────────────────────────────────────────────────────────────
def time_based_split(df, X, y, groups,
                     train_end=2022, valid_year=2023, test_year=2024):
    """
    Temporal split with no shuffling.
    - train  : Year <= train_end
    - valid  : Year == valid_year   (used for Optuna objective)
    - test   : Year == test_year    (held-out; touched only for final evaluation)
    Groups are race-level (Year_EventName) so CV folds never leak across races.
    """
    train_mask = df["Year"] <= train_end
    valid_mask = df["Year"] == valid_year
    test_mask  = df["Year"] == test_year
    return {
        "X_train":      X[train_mask],   "y_train":      y[train_mask],
        "X_valid":      X[valid_mask],   "y_valid":      y[valid_mask],
        "X_test":       X[test_mask],    "y_test":       y[test_mask],
        "groups_train": groups[train_mask],
        "groups_valid": groups[valid_mask],
        "groups_test":  groups[test_mask],
    }

def dataset_model(dataset, drop_columns):
    """Return a copy of dataset with the given columns removed."""
    return dataset.drop(columns=drop_columns)


In [ ]:
# ── EDA helpers ───────────────────────────────────────────────────────────────
def dataset_eda(df):
    """Shape, missing-value table, numeric histograms."""
    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)
    print(f"Shape: {df.shape}")
    missing = df.isna().sum().sort_values(ascending=False)
    missing = missing[missing > 0]
    if len(missing):
        print("\nMissing Values:")
        display(missing.to_frame("Missing"))
    df.select_dtypes(include="number").hist(bins=30, figsize=(18, 12))
    plt.tight_layout(); plt.show()

def regression_target_eda(df, target):
    """Describe + distribution plot + feature correlation heatmap."""
    print("=" * 60)
    print(f"TARGET EDA: {target}")
    print("=" * 60)
    display(df[target].describe())
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(df[target], kde=True, ax=ax[0]).set_title("Target Distribution")
    sns.boxplot(x=df[target], ax=ax[1]).set_title("Outlier Check")
    plt.tight_layout(); plt.show()
    corr = df.select_dtypes(include="number").corr()[target].sort_values(ascending=False).drop(target)
    plt.figure(figsize=(8, max(6, len(corr) * 0.3)))
    sns.heatmap(corr.to_frame(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title(f"Feature Correlation with {target}"); plt.show()

def classification_target_eda(df, target):
    """Class-count bar + positive-class rate for binary targets."""
    counts = df[target].value_counts()
    display(counts)
    sns.countplot(x=df[target]); plt.title(target); plt.show()
    print(f"Positive Class Rate: {counts.get(1,0)/len(df):.2%}")


In [ ]:
# ── Evaluation & visualisation helpers ───────────────────────────────────────
def evaluate_regression(model, X, y, dataset_name="Test"):
    """Print RMSE / MAE / R² and return predictions."""
    pred = model.predict(X)
    print(f"{dataset_name} RMSE : {root_mean_squared_error(y, pred):.3f}")
    print(f"{dataset_name} MAE  : {mean_absolute_error(y, pred):.3f}")
    print(f"{dataset_name} R²   : {r2_score(y, pred):.3f}")
    return pred

def plot_residuals(y_true, y_pred):
    residuals = y_true - y_pred
    plt.figure(figsize=(8, 5))
    plt.hist(residuals, bins=50)
    plt.xlabel("Residual"); plt.ylabel("Count")
    plt.title("Residual Distribution"); plt.show()

def prediction_plot(y_true, y_pred):
    plt.figure(figsize=(7, 7))
    plt.scatter(y_pred, y_true, alpha=0.2)
    lo, hi = y_true.min(), y_true.max()
    plt.plot([lo, hi], [lo, hi], "r--")
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

def shap_analysis(model, X_transformed, feature_names=None, max_samples=5000):
    """Bar + beeswarm SHAP summary on already-transformed data."""
    idx      = np.random.choice(len(X_transformed), min(max_samples, len(X_transformed)), replace=False)
    X_sample = X_transformed[idx] if hasattr(X_transformed, "__getitem__") else X_transformed.iloc[idx]
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_sample)
    shap.summary_plot(shap_vals, X_sample, feature_names=feature_names, plot_type="bar")
    shap.summary_plot(shap_vals, X_sample, feature_names=feature_names)
    return explainer, shap_vals, X_sample


---
## 2 · Data Loading & Feature Engineering

Load the raw dataset, compute rolling/lag, sector-delta, track-position and gap features, then persist the enriched CSV.

In [ ]:
dataset = pd.read_csv("/content/f1_strategy_final_dataset.csv")
print(f"Raw dataset: {dataset.shape}")
dataset.head(3)


In [ ]:
# ── Sort & rolling / lag lap-time features ────────────────────────────────────
# All transforms are shifted by ≥1 lap within the driver-race group → zero leakage.
dataset = dataset.sort_values(["Year", "EventName", "Driver", "LapNumber"])
dg = dataset.groupby(["Year", "EventName", "Driver"])

dataset["Rolling3LapTime"] = dg["LapTimeSec"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
dataset["Rolling5LapTime"] = dg["LapTimeSec"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
dataset["LapTimeDelta"]    = dg["LapTimeSec"].shift(1).diff().fillna(0)
dataset["PrevLapTime"]     = dg["LapTimeSec"].shift(1)

for s in ["Sector1TimeSec", "Sector2TimeSec", "Sector3TimeSec"]:
    dataset[f"{s}_Delta"] = dg[s].shift(1).diff().fillna(0)

print("Rolling / lag features added ✓")


In [ ]:
# ── Track-position & gap features ─────────────────────────────────────────────
dataset["PositionGain"] = dataset["GridPosition"] - dataset["Position"]

field_median = dataset.groupby(["Year", "EventName", "LapNumber"])["LapTimeSec"].transform("median")
dataset["PrevFieldMedian"] = (
    field_median.groupby([dataset["Year"], dataset["EventName"], dataset["Driver"]]).shift(1)
)
dataset["LapTimeVsField"] = dataset["PrevLapTime"] - dataset["PrevFieldMedian"]

# Cumulative race time → on-track gap (lagged 1 lap to prevent leakage)
dataset = dataset.sort_values(["Year", "EventName", "Driver", "LapNumber"])
dataset["RaceTime"] = dataset.groupby(["Year", "EventName", "Driver"])["LapTimeSec"].cumsum()

dataset = dataset.sort_values(["Year", "EventName", "LapNumber", "Position"])
lg = dataset.groupby(["Year", "EventName", "LapNumber"])
dataset["GapAhead"]  = dataset["RaceTime"] - lg["RaceTime"].shift(1)
dataset["GapBehind"] = lg["RaceTime"].shift(-1) - dataset["RaceTime"]

for col in ["GapAhead", "GapBehind"]:
    dataset[col] = dataset.groupby(["Year", "EventName", "Driver"])[col].shift(1)

dataset["GapAhead"]  = dataset["GapAhead"].fillna(999)
dataset["GapBehind"] = dataset["GapBehind"].fillna(999)

last_pos = dataset.groupby(["Year", "EventName", "LapNumber"])["Position"].transform("max")
dataset["IsLeader"] = (dataset["Position"] == 1).astype(int)
dataset["IsLast"]   = (dataset["Position"] == last_pos).astype(int)

print("Position / gap features added ✓")


In [ ]:
dataset.to_csv("F1_Strategy_Prediction_Dataset.csv", index=False)
print(f"Enriched dataset saved  shape={dataset.shape}")
dataset.head(3)


---
## 3 · Shared CV & Column Masks

Race-level group key, 5-fold `GroupKFold`, and drop-column masks for each model — defined once to avoid drift.

In [ ]:
groups = dataset["Year"].astype(str) + "_" + dataset["EventName"]
gkf    = GroupKFold(n_splits=5)

# Drop masks — each model sees only its relevant columns
DROP_M1   = ["Sector1TimeSec", "Sector2TimeSec", "Sector3TimeSec",
              "LapsUntilNextPit", "IsPersonalBest", "IsPitLap"]

DROP_M2   = ["LapsUntilNextPit", "LapTimeSec",
              "Sector1TimeSec", "Sector2TimeSec", "Sector3TimeSec", "IsPersonalBest"]

DROP_M345 = ["IsPitLap", "LapTimeSec",
              "Sector1TimeSec", "Sector2TimeSec", "Sector3TimeSec", "IsPersonalBest"]

print("Groups, CV splitter, and drop masks ready ✓")


---
## Model 1 · Lap-Time Regressor (LightGBM + Optuna)

**Target:** `LapTimeSec`  
**Algorithm:** LightGBM · Optuna TPE · 50 trials · minimise validation RMSE  
**Achieved:** Test RMSE **4.298** · MAE **1.971**


In [ ]:
# ── 1-a  Dataset ──────────────────────────────────────────────────────────────
df_m1 = dataset_model(dataset, DROP_M1)
regression_target_eda(df_m1, "LapTimeSec")

X_1 = df_m1.drop(columns=["LapTimeSec"])
y_1 = df_m1["LapTimeSec"]

splits_m1    = time_based_split(df_m1, X_1, y_1, groups)
X_train      = splits_m1["X_train"];   y_train      = splits_m1["y_train"]
X_valid      = splits_m1["X_valid"];   y_valid      = splits_m1["y_valid"]
X_test       = splits_m1["X_test"];    y_test       = splits_m1["y_test"]
groups_train = splits_m1["groups_train"]


In [ ]:
# ── 1-b  Feature engineering transformer ─────────────────────────────────────
class FeatureEngineeringM1(BaseEstimator, TransformerMixin):
    """
    Learns max laps per race during fit (prevents FuelLoad leakage at transform time).
    Adds: FuelLoad, TrackAirDelta, TyreDegRate, TrackWind, CompoundDeg.
    """
    def fit(self, X, y=None):
        self.max_laps_ = X.groupby(["Year", "EventName"])["LapNumber"].max()
        return self

    def transform(self, X):
        X = X.copy()
        X["FuelLoad"]      = X.apply(
            lambda r: self.max_laps_.get((r["Year"], r["EventName"]), r["LapNumber"]) - r["LapNumber"],
            axis=1
        )
        X["TrackAirDelta"] = X["TrackTemp"]  - X["AirTemp"]
        X["TyreDegRate"]   = X["TyreLife"]   / (X["LapNumber"] + 1)
        X["TrackWind"]     = X["TrackTemp"]  * X["WindSpeed"]
        X["CompoundDeg"]   = X["TyreLife"]   * X["CompoundCode"]
        X["Rainfall"]      = X["Rainfall"].astype(int)
        return X


In [ ]:
# ── 1-c  Preprocessing pipeline ──────────────────────────────────────────────
num_features_m1 = [
    "LapNumber", "Stint", "TyreLife", "GridPosition",
    "AirTemp", "TrackTemp", "WindSpeed", "IsSC", "IsDRS",
    "TyreDegRate", "CompoundDeg", "Rolling3LapTime", "Rolling5LapTime",
    "LapTimeDelta", "LapTimeVsField",
    "Sector1TimeSec_Delta", "Sector2TimeSec_Delta", "Sector3TimeSec_Delta",
    "PrevLapTime", "Rainfall", "IsLast", "IsLeader", "GapAhead", "GapBehind",
    "FuelLoad", "TrackAirDelta", "TrackWind",
]
cat_features_m1 = ["Compound", "Team", "Driver", "EventName"]

preprocessing_m1 = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
    ]), num_features_m1),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", TargetEncoder(target_type="continuous", random_state=42, smooth=10.0)),
    ]), cat_features_m1),
])


In [ ]:
# ── 1-d  Optuna hyperparameter search ─────────────────────────────────────────
def objective_m1(trial):
    params = {
        "n_estimators":      trial.suggest_int(  "n_estimators",       300, 2000),
        "learning_rate":     trial.suggest_float( "learning_rate",     0.005, 0.15, log=True),
        "num_leaves":        trial.suggest_int(  "num_leaves",          15,  255),
        "max_depth":         trial.suggest_int(  "max_depth",            3,   16),
        "min_child_samples": trial.suggest_int(  "min_child_samples",    5,  100),
        "subsample":         trial.suggest_float( "subsample",          0.5,  1.0),
        "colsample_bytree":  trial.suggest_float( "colsample_bytree",   0.5,  1.0),
        "reg_alpha":         trial.suggest_float( "reg_alpha",         1e-5,  10,  log=True),
        "reg_lambda":        trial.suggest_float( "reg_lambda",        1e-5,  10,  log=True),
        "random_state": 42, "n_jobs": -1, "verbosity": -1,
    }
    pipe = Pipeline([
        ("fe",    FeatureEngineeringM1()),
        ("pre",   preprocessing_m1),
        ("model", LGBMRegressor(**params)),
    ])
    pipe.fit(X_train, y_train)
    return root_mean_squared_error(y_valid, pipe.predict(X_valid))

study_m1 = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
study_m1.optimize(objective_m1, n_trials=50, show_progress_bar=True)

best_params_m1 = study_m1.best_params
print(f"\nBest Validation RMSE : {study_m1.best_value:.4f}")
print(f"Best Params:\n{best_params_m1}")


In [ ]:
# ── 1-e  Final model — train on train+valid, evaluate on test ─────────────────
X1_final = X_1[df_m1["Year"] <= 2023]
y1_final = y_1[df_m1["Year"] <= 2023]

model1_pipeline = Pipeline([
    ("fe",    FeatureEngineeringM1()),
    ("pre",   preprocessing_m1),
    ("model", LGBMRegressor(**best_params_m1, random_state=42, n_jobs=-1, verbosity=-1)),
])
model1_pipeline.fit(X1_final, y1_final)

y_pred_m1 = model1_pipeline.predict(X_test)
print("── M1 Test Scores ─────────────────────────────────────")
print(f"  RMSE : {root_mean_squared_error(y_test, y_pred_m1):.3f}")
print(f"  MAE  : {mean_absolute_error(y_test, y_pred_m1):.3f}")


In [ ]:
# ── 1-f  Residuals & SHAP ────────────────────────────────────────────────────
plot_residuals(y_test, y_pred_m1)
prediction_plot(y_test, y_pred_m1)

X_test_fe  = model1_pipeline.named_steps["fe"].transform(X_test)
X_test_pre = model1_pipeline.named_steps["pre"].transform(X_test_fe)
shap_analysis(
    model1_pipeline.named_steps["model"],
    X_test_pre,
    feature_names=num_features_m1 + cat_features_m1,
)


---
## Model 2 · Pit-Lap Classifier (CatBoost + Optuna)

**Target:** `IsPitLap` — binary (1 = driver pits this lap)  
**Class imbalance:** ~3.1 % positive → `auto_class_weights="Balanced"`  
**Algorithm:** CatBoostClassifier · Optuna TPE · 50 trials · maximise ROC-AUC  
**Achieved:** Test ROC-AUC **0.8226** · PR-AUC **0.1439**

> OOF pit-probabilities from this model become the `PitProb` meta-feature in M3 & M4.


In [ ]:
# ── 2-a  Dataset ──────────────────────────────────────────────────────────────
df_m2 = dataset_model(dataset, DROP_M2)
classification_target_eda(df_m2, "IsPitLap")

X_2 = df_m2.drop(columns=["IsPitLap"])
y_2 = df_m2["IsPitLap"]

splits_m2    = time_based_split(df_m2, X_2, y_2, groups)
X_train      = splits_m2["X_train"];   y_train      = splits_m2["y_train"]
X_valid      = splits_m2["X_valid"];   y_valid      = splits_m2["y_valid"]
X_test       = splits_m2["X_test"];    y_test       = splits_m2["y_test"]
groups_train = splits_m2["groups_train"]


In [ ]:
# ── 2-b  Feature engineering transformer ─────────────────────────────────────
class FeatureEngineeringM2(BaseEstimator, TransformerMixin):
    """
    Tyre-wear, race-progress, and pit-pressure features.
    Includes a fillna guard for Compound before dictionary lookup.
    """
    EXPECTED_LIFE = {"SOFT": 18, "MEDIUM": 28, "HARD": 38, "INTERMEDIATE": 25, "WET": 30}

    def fit(self, X, y=None): return self

    def transform(self, X):
        X = X.copy()
        X["Compound"] = X["Compound"].fillna("UNKNOWN_COMPOUND")
        race_len = X.groupby(["Year", "EventName"])["LapNumber"].transform("max")
        X["FuelLoad"]          = race_len - X["LapNumber"]
        X["TrackAirDelta"]     = X["TrackTemp"] - X["AirTemp"]
        X["TyreDegRate"]       = X["TyreLife"] / (X["LapNumber"] + 1)
        X["TrackWind"]         = X["TrackTemp"] * X["WindSpeed"]
        X["CompoundDeg"]       = X["TyreLife"] * X["CompoundCode"]
        X["Rainfall"]          = X["Rainfall"].astype(int)
        X["ExpectedTyreLife"]  = X["Compound"].map(self.EXPECTED_LIFE).fillna(25)
        X["TyreWearPct"]       = X["TyreLife"] / X["ExpectedTyreLife"]
        X["RaceProgress"]      = X["LapNumber"] / race_len
        X["CompoundTyreLife"]  = X["CompoundCode"] * X["TyreLife"]
        X["FinalStint"]        = (X["Stint"] >= 3).astype(int)
        X["PitPressure"]       = X["TyreWearPct"] * X["RaceProgress"]
        X["PositionDelta"]     = X["Position"] - X["GridPosition"]
        X["TeamCompound"]      = X["Team"] + "_" + X["Compound"].astype(str)
        X["LapsRemaining"]     = race_len - X["LapNumber"]
        X["WearRemaining"]     = X["TyreWearPct"] * X["LapsRemaining"]
        return X


In [ ]:
# ── 2-c  Preprocessing pipeline ──────────────────────────────────────────────
num_features_m2 = [
    "LapNumber", "Stint", "TrackTemp", "Position", "IsSC", "IsDRS",
    "GapAhead", "GapBehind", "IsLast", "IsLeader", "RaceTime",
    "FuelLoad", "TyreDegRate", "CompoundDeg", "TyreWearPct",
    "RaceProgress", "ExpectedTyreLife", "PitPressure",
    "LapsRemaining", "WearRemaining", "CompoundTyreLife", "FinalStint", "PositionDelta",
]
cat_features_m2 = ["Compound", "EventName", "Team"]
bin_features_m2 = ["FreshTyre"]

preprocessing_m2 = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scl", StandardScaler()),
    ]), num_features_m2),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", TargetEncoder(target_type="binary", random_state=42, smooth=10.0)),
    ]), cat_features_m2),
    ("bin", "passthrough", bin_features_m2),
])


In [ ]:
# ── 2-d  Optuna hyperparameter search ─────────────────────────────────────────
def objective_m2(trial):
    params = {
        "iterations":          trial.suggest_int(  "iterations",          400,  900),
        "depth":               trial.suggest_int(  "depth",                 5,    8),
        "learning_rate":       trial.suggest_float( "learning_rate",      0.03,  0.10, log=True),
        "l2_leaf_reg":         trial.suggest_float( "l2_leaf_reg",         3.0,  10.0),
        "random_strength":     trial.suggest_float( "random_strength",     0.0,   3.0),
        "bagging_temperature": trial.suggest_float( "bagging_temperature", 0.0,   3.0),
        "border_count": 128,
        "loss_function": "Logloss",
        "auto_class_weights": "Balanced",
        "eval_metric": "AUC",
        "random_state": 42, "verbose": 0,
    }
    pipe = Pipeline([
        ("fe",    FeatureEngineeringM2()),
        ("pre",   preprocessing_m2),
        ("model", CatBoostClassifier(**params)),
    ])
    pipe.fit(X_train, y_train)
    return roc_auc_score(y_valid, pipe.predict_proba(X_valid)[:, 1])

study_m2 = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=42))
study_m2.optimize(objective_m2, n_trials=50, show_progress_bar=True)

best_params_m2 = study_m2.best_params
print(f"\nBest Validation ROC-AUC : {study_m2.best_value:.4f}")
print(f"Best Params:\n{best_params_m2}")


In [ ]:
# ── 2-e  Final model — fit on train, evaluate on test ────────────────────────
final_params_m2 = {
    **best_params_m2,
    "border_count": 128,
    "loss_function": "Logloss",
    "auto_class_weights": "Balanced",
    "eval_metric": "PRAUC",
    "random_state": 42, "verbose": 0,
}

m2_pipeline = Pipeline([
    ("fe",    FeatureEngineeringM2()),
    ("pre",   preprocessing_m2),
    ("model", CatBoostClassifier(**final_params_m2)),
])
m2_pipeline.fit(X_train, y_train)

proba_test_m2 = m2_pipeline.predict_proba(X_test)[:, 1]
proba_val_m2  = m2_pipeline.predict_proba(X_valid)[:, 1]

print("── M2 Test Scores ─────────────────────────────────────")
print(f"  ROC-AUC : {roc_auc_score(y_test, proba_test_m2):.4f}")
print(f"  PR-AUC  : {average_precision_score(y_test, proba_test_m2):.4f}")


In [ ]:
# ── 2-f  Threshold analysis on validation set ─────────────────────────────────
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_valid, proba_val_m2)
f1_arr   = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
best_idx = np.argmax(f1_arr)

print(f"Best F1 threshold : {thresh_arr[best_idx]:.4f}")
print(f"  F1        : {f1_arr[best_idx]:.4f}")
print(f"  Precision : {prec_arr[best_idx]:.4f}")
print(f"  Recall    : {rec_arr[best_idx]:.4f}")

for name, thr in [("Aggressive", 0.25), ("Balanced", 0.49), ("Conservative", 0.80)]:
    pred = (proba_val_m2 >= thr).astype(int)
    print(f"  {name:13s} thr={thr:.2f} | "
          f"P={precision_score(y_valid, pred, zero_division=0):.3f} "
          f"R={recall_score(y_valid, pred):.3f} "
          f"F1={f1_score(y_valid, pred):.3f}")

PrecisionRecallDisplay.from_predictions(y_test, proba_test_m2)
plt.title("M2 — Precision-Recall Curve (test set)"); plt.show()


In [ ]:
# ── 2-g  OOF pit-probabilities (meta-feature for M3 & M4) ────────────────────
oof_pit_prob   = np.zeros(len(X_train))
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
    fold_pipe = Pipeline([
        ("fe",    FeatureEngineeringM2()),
        ("pre",   preprocessing_m2),
        ("model", CatBoostClassifier(**final_params_m2)),
    ])
    fold_pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    oof_pit_prob[val_idx] = fold_pipe.predict_proba(X_train.iloc[val_idx])[:, 1]

# Retrain on full train set for valid/test estimates
m2_pipeline.fit(X_train, y_train)
valid_pit_prob = m2_pipeline.predict_proba(X_valid)[:, 1]
test_pit_prob  = m2_pipeline.predict_proba(X_test)[:, 1]

print("OOF pit-probabilities computed ✓")
print(f"  OOF ROC-AUC (sanity) : {roc_auc_score(y_train, oof_pit_prob):.4f}")


---
## Model 3 · Pit-in-3-Laps Classifier (CatBoost + Optuna)

**Target:** `WillPitWithin3Laps` — 1 if `LapsUntilNextPit ∈ {1, 2, 3}`  
**Class imbalance:** ~8.7 % positive  
**Meta-feature:** `PitProb` (OOF from M2, index-aligned)  
**Algorithm:** CatBoostClassifier · Optuna TPE · 50 trials · maximise ROC-AUC  
**Achieved:** Test ROC-AUC **0.7856** · PR-AUC **0.2606** · F1 **0.3308**


In [ ]:
# ── 3-a  Dataset ──────────────────────────────────────────────────────────────
df_m3 = dataset_model(dataset, DROP_M345).copy()
df_m3["WillPitWithin3Laps"] = df_m3["LapsUntilNextPit"].between(1, 3).astype(int)
classification_target_eda(df_m3, "WillPitWithin3Laps")

X_3 = df_m3.drop(columns=["LapsUntilNextPit", "WillPitWithin3Laps",
                            "DriverNumber", "RoundNumber"])
y_3 = df_m3["WillPitWithin3Laps"]

splits_m3    = time_based_split(df_m3, X_3, y_3, groups)
X_train      = splits_m3["X_train"].copy();  y_train      = splits_m3["y_train"]
X_valid      = splits_m3["X_valid"].copy();  y_valid      = splits_m3["y_valid"]
X_test       = splits_m3["X_test"].copy();   y_test       = splits_m3["y_test"]
groups_train = splits_m3["groups_train"]

# Attach PitProb meta-feature (index-aligned OOF from M2)
oof_s   = pd.Series(oof_pit_prob,   index=splits_m2["X_train"].index)
valid_s = pd.Series(valid_pit_prob, index=splits_m2["X_valid"].index)
test_s  = pd.Series(test_pit_prob,  index=splits_m2["X_test"].index)

X_train["PitProb"] = oof_s.reindex(X_train.index)
X_valid["PitProb"] = valid_s.reindex(X_valid.index)
X_test["PitProb"]  = test_s.reindex(X_test.index)
print("PitProb meta-feature attached ✓")


In [ ]:
# ── 3-b  Feature engineering transformer ─────────────────────────────────────
class FeatureEngineeringM3(BaseEstimator, TransformerMixin):
    EXPECTED_LIFE = {"SOFT": 18, "MEDIUM": 28, "HARD": 38, "INTERMEDIATE": 25, "WET": 30}

    def fit(self, X, y=None): return self

    def transform(self, X):
        X = X.copy()
        race_len = X.groupby(["Year", "EventName"])["LapNumber"].transform("max")
        X["FuelLoad"]          = race_len - X["LapNumber"]
        X["TrackAirDelta"]     = X["TrackTemp"] - X["AirTemp"]
        X["TyreDegRate"]       = X["TyreLife"] / (X["LapNumber"] + 1)
        X["TrackWind"]         = X["TrackTemp"] * X["WindSpeed"]
        X["CompoundDeg"]       = X["TyreLife"] * X["CompoundCode"]
        X["Rainfall"]          = X["Rainfall"].astype(int)
        X["ExpectedTyreLife"]  = X["Compound"].map(self.EXPECTED_LIFE).fillna(25)
        X["TyreWearPct"]       = X["TyreLife"] / X["ExpectedTyreLife"]
        X["RaceProgress"]      = X["LapNumber"] / race_len
        X["CompoundTyreLife"]  = X["CompoundCode"] * X["TyreLife"]
        X["FinalStint"]        = (X["Stint"] >= 3).astype(int)
        X["PitPressure"]       = X["TyreWearPct"] * X["RaceProgress"]
        X["PositionDelta"]     = X["Position"] - X["GridPosition"]
        X["TeamCompound"]      = X["Team"] + "_" + X["Compound"].astype(str)
        X["LapsToExpectedEnd"] = X["ExpectedTyreLife"] - X["TyreLife"]
        X["TyreExceededLife"]  = (X["TyreLife"] >= X["ExpectedTyreLife"]).astype(int)
        return X


In [ ]:
# ── 3-c  Preprocessing pipeline ──────────────────────────────────────────────
num_features_m3 = [
    "LapNumber", "Stint", "TrackTemp", "TyreLife", "IsSC", "IsVSC",
    "GridPosition", "AirTemp", "WindSpeed", "PitProb",
    "FuelLoad", "TyreDegRate", "RaceProgress", "PitPressure",
    "ExpectedTyreLife", "TyreWearPct",
    "Rolling3LapTime", "Rolling5LapTime", "LapTimeDelta", "PositionGain",
    "RaceTime", "GapAhead", "GapBehind",
]
cat_features_m3 = [
    "Compound", "EventName", "Team", "Driver",
    "Rainfall", "FreshTyre", "FinalStint", "IsLeader", "IsLast",
]

preprocessing_m3 = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scl", StandardScaler()),
    ]), num_features_m3),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", TargetEncoder(target_type="binary", random_state=42, smooth=10.0)),
    ]), cat_features_m3),
])


In [ ]:
# ── 3-d  Optuna hyperparameter search ─────────────────────────────────────────
def objective_m3(trial):
    params = {
        "iterations":          trial.suggest_int(  "iterations",          400,  900),
        "depth":               trial.suggest_int(  "depth",                 5,    8),
        "learning_rate":       trial.suggest_float( "learning_rate",      0.03,  0.10, log=True),
        "l2_leaf_reg":         trial.suggest_float( "l2_leaf_reg",         3.0,  10.0),
        "random_strength":     trial.suggest_float( "random_strength",     0.0,   3.0),
        "bagging_temperature": trial.suggest_float( "bagging_temperature", 0.0,   3.0),
        "border_count": 128,
        "loss_function": "Logloss",
        "auto_class_weights": "Balanced",
        "eval_metric": "AUC",
        "random_state": 42, "verbose": 0,
    }
    pipe = Pipeline([
        ("fe",    FeatureEngineeringM3()),
        ("pre",   preprocessing_m3),
        ("model", CatBoostClassifier(**params)),
    ])
    pipe.fit(X_train, y_train)
    return roc_auc_score(y_valid, pipe.predict_proba(X_valid)[:, 1])

study_m3 = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=42))
study_m3.optimize(objective_m3, n_trials=50, show_progress_bar=True)

best_params_m3 = study_m3.best_params
print(f"\nBest Validation ROC-AUC : {study_m3.best_value:.4f}")
print(f"Best Params:\n{best_params_m3}")


In [ ]:
# ── 3-e  Final model — fit on train, evaluate on test ────────────────────────
final_params_m3 = {
    **best_params_m3,
    "border_count": 128,
    "loss_function": "Logloss",
    "auto_class_weights": "Balanced",
    "eval_metric": "PRAUC",
    "random_state": 42, "verbose": 0,
}

m3_pipeline = Pipeline([
    ("fe",    FeatureEngineeringM3()),
    ("pre",   preprocessing_m3),
    ("model", CatBoostClassifier(**final_params_m3)),
])
m3_pipeline.fit(X_train, y_train)

proba_test_m3 = m3_pipeline.predict_proba(X_test)[:, 1]
preds_test_m3 = (proba_test_m3 >= 0.5).astype(int)

print("── M3 Test Scores ─────────────────────────────────────")
print(f"  ROC-AUC   : {roc_auc_score(y_test, proba_test_m3):.4f}")
print(f"  PR-AUC    : {average_precision_score(y_test, proba_test_m3):.4f}")
print(f"  Precision : {precision_score(y_test, preds_test_m3):.4f}")
print(f"  Recall    : {recall_score(y_test, preds_test_m3):.4f}")
print(f"  F1        : {f1_score(y_test, preds_test_m3):.4f}")


In [ ]:
# ── 3-f  OOF pit-in-3 probabilities (meta-feature for M4) ────────────────────
oof_pit_prob_m3 = np.zeros(len(X_train))
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
    fold_pipe = Pipeline([
        ("fe",    FeatureEngineeringM3()),
        ("pre",   preprocessing_m3),
        ("model", CatBoostClassifier(**final_params_m3)),
    ])
    fold_pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    oof_pit_prob_m3[val_idx] = fold_pipe.predict_proba(X_train.iloc[val_idx])[:, 1]

m3_pipeline.fit(X_train, y_train)
valid_pit_prob_m3 = m3_pipeline.predict_proba(X_valid)[:, 1]
test_pit_prob_m3  = m3_pipeline.predict_proba(X_test)[:, 1]

print("OOF pit-in-3 probabilities computed ✓")


---
## Model 4 · Long-Horizon Laps-Until-Pit Regressor (CatBoost + Optuna)

**Target:** `LapsUntilNextPit` — all laps (excluding immediate pit laps)  
**Meta-feature:** `PitProbIn3` (OOF from M3)  
**Algorithm:** CatBoostRegressor · Optuna TPE · 50 trials · minimise MAE · log1p target  
**Achieved:** OOF RMSE **7.248** · Test RMSE **7.677** · MAE **5.503**


In [ ]:
# ── 4-a  Dataset ──────────────────────────────────────────────────────────────
df_m4 = dataset_model(dataset, DROP_M345)
df_m4 = df_m4[df_m4["LapsUntilNextPit"] != 1]          # exclude laps where car is already pitting
regression_target_eda(df_m4, "LapsUntilNextPit")

X_4 = df_m4.drop(columns=["LapsUntilNextPit"])
y_4 = df_m4["LapsUntilNextPit"]

groups_m4    = df_m4["Year"].astype(str) + "_" + df_m4["EventName"]
splits_m4    = time_based_split(df_m4, X_4, y_4, groups_m4)
X_train      = splits_m4["X_train"].copy();  y_train      = splits_m4["y_train"]
X_valid      = splits_m4["X_valid"].copy();  y_valid      = splits_m4["y_valid"]
X_test       = splits_m4["X_test"].copy();   y_test       = splits_m4["y_test"]
groups_train = splits_m4["groups_train"]

# Attach PitProbIn3 meta-feature (OOF from M3, index-aligned)
oof3_s   = pd.Series(oof_pit_prob_m3,   index=splits_m3["X_train"].index)
valid3_s = pd.Series(valid_pit_prob_m3, index=splits_m3["X_valid"].index)
test3_s  = pd.Series(test_pit_prob_m3,  index=splits_m3["X_test"].index)

X_train["PitProbIn3"] = oof3_s.reindex(X_train.index)
X_valid["PitProbIn3"] = valid3_s.reindex(X_valid.index)
X_test["PitProbIn3"]  = test3_s.reindex(X_test.index)
print("PitProbIn3 meta-feature attached ✓")


In [ ]:
# ── 4-b  Feature engineering transformer ─────────────────────────────────────
class FeatureEngineeringM4(BaseEstimator, TransformerMixin):
    """Extended tyre-wear + race-progress features including tyre-life flags."""
    EXPECTED_LIFE = {"SOFT": 18, "MEDIUM": 28, "HARD": 38, "INTERMEDIATE": 25, "WET": 30}

    def fit(self, X, y=None): return self

    def transform(self, X):
        X = X.copy()
        race_len = X.groupby(["Year", "EventName"])["LapNumber"].transform("max")
        X["FuelLoad"]          = race_len - X["LapNumber"]
        X["TrackAirDelta"]     = X["TrackTemp"] - X["AirTemp"]
        X["TyreDegRate"]       = X["TyreLife"] / (X["LapNumber"] + 1)
        X["TrackWind"]         = X["TrackTemp"] * X["WindSpeed"]
        X["CompoundDeg"]       = X["TyreLife"] * X["CompoundCode"]
        X["Rainfall"]          = X["Rainfall"].astype(int)
        X["ExpectedTyreLife"]  = X["Compound"].map(self.EXPECTED_LIFE).fillna(25)
        X["TyreWearPct"]       = X["TyreLife"] / X["ExpectedTyreLife"]
        X["RaceProgress"]      = X["LapNumber"] / race_len
        X["CompoundTyreLife"]  = X["CompoundCode"] * X["TyreLife"]
        X["FinalStint"]        = (X["Stint"] >= 3).astype(int)
        X["PitPressure"]       = X["TyreWearPct"] * X["RaceProgress"]
        X["PositionDelta"]     = X["Position"] - X["GridPosition"]
        X["TeamCompound"]      = X["Team"] + "_" + X["Compound"].astype(str)
        X["LapsToExpectedEnd"] = X["ExpectedTyreLife"] - X["TyreLife"]
        X["TyreExceededLife"]  = (X["TyreLife"] >= X["ExpectedTyreLife"]).astype(int)
        return X


In [ ]:
# ── 4-c  Preprocessing pipeline ──────────────────────────────────────────────
num_features_m4 = [
    "LapNumber", "Stint", "TyreLife", "CompoundCode", "IsSC", "IsVSC",
    "GridPosition", "FuelLoad", "TyreDegRate", "WindSpeed",
    "RaceProgress", "PitPressure", "ExpectedTyreLife", "TyreWearPct",
    "Rolling3LapTime", "Rolling5LapTime", "LapTimeDelta", "PositionGain",
    "RaceTime", "GapAhead", "GapBehind", "LapsToExpectedEnd", "PitProbIn3",
]
cat_features_m4 = ["EventName", "Team", "FreshTyre", "FinalStint", "IsLeader", "IsLast"]

preprocessing_m4 = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scl", StandardScaler()),
    ]), num_features_m4),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", TargetEncoder(target_type="continuous", random_state=42, smooth=10.0)),
    ]), cat_features_m4),
])


In [ ]:
# ── 4-d  Optuna hyperparameter search ─────────────────────────────────────────
def objective_m4(trial):
    params = {
        "loss_function": "MAE", "eval_metric": "MAE",
        "iterations": 3000,
        "depth":               trial.suggest_int(  "depth",                4,  10),
        "learning_rate":       trial.suggest_float( "learning_rate",      0.01, 0.15, log=True),
        "l2_leaf_reg":         trial.suggest_float( "l2_leaf_reg",         1.0, 30.0, log=True),
        "random_strength":     trial.suggest_float( "random_strength",     0.0, 10.0),
        "bagging_temperature": trial.suggest_float( "bagging_temperature", 0.0, 10.0),
        "min_data_in_leaf":    trial.suggest_int(  "min_data_in_leaf",     5,  100),
        "grow_policy":         trial.suggest_categorical("grow_policy",
                                                          ["SymmetricTree", "Depthwise"]),
        "random_seed": 42, "verbose": 0,
    }
    pipe = Pipeline([
        ("fe",    FeatureEngineeringM4()),
        ("pre",   preprocessing_m4),
        ("model", CatBoostRegressor(**params)),
    ])
    pipe.fit(X_train, y_train)
    return mean_absolute_error(y_valid, pipe.predict(X_valid))

study_m4 = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
study_m4.optimize(objective_m4, n_trials=50, show_progress_bar=True)

best_params_m4 = study_m4.best_params
print(f"\nBest Validation MAE : {study_m4.best_value:.4f}")
print(f"Best Params:\n{best_params_m4}")


In [ ]:
# ── 4-e  5-fold OOF training ─────────────────────────────────────────────────
oof_preds_m4 = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_vl, y_vl = X_train.iloc[val_idx], y_train.iloc[val_idx]

    fold_pipe = Pipeline([
        ("fe",    FeatureEngineeringM4()),
        ("pre",   preprocessing_m4),
        ("model", CatBoostRegressor(
            iterations=3000, loss_function="MAE", eval_metric="MAE",
            **best_params_m4
        )),
    ])
    fold_pipe.fit(X_tr, y_tr)
    preds_fold           = fold_pipe.predict(X_vl)
    oof_preds_m4[val_idx] = preds_fold

    print(f"  Fold {fold+1} | MAE={mean_absolute_error(y_vl, preds_fold):.4f} "
          f"| RMSE={root_mean_squared_error(y_vl, preds_fold):.4f}")

print(f"\n── M4 OOF ──────────────────────────────────────────────")
print(f"  OOF MAE  : {mean_absolute_error(y_train, oof_preds_m4):.4f}")
print(f"  OOF RMSE : {root_mean_squared_error(y_train, oof_preds_m4):.4f}")


In [ ]:
# ── 4-f  Final model — log1p target, evaluate on test ─────────────────────────
m4_pipeline = Pipeline([
    ("fe",    FeatureEngineeringM4()),
    ("pre",   preprocessing_m4),
    ("model", CatBoostRegressor(
        iterations=3000, loss_function="MAE", eval_metric="MAE",
        **best_params_m4
    )),
])
y_log = np.log1p(y_train)
m4_pipeline.fit(X_train, y_log)

preds_m4 = np.expm1(m4_pipeline.predict(X_test))

print("── M4 Test Scores ─────────────────────────────────────")
print(f"  MAE  : {mean_absolute_error(y_test, preds_m4):.4f}")
print(f"  RMSE : {root_mean_squared_error(y_test, preds_m4):.4f}")


In [ ]:
# ── 4-g  Error breakdown by horizon bucket ────────────────────────────────────
results_m4 = pd.DataFrame({"actual": y_test, "pred": preds_m4})
results_m4["bucket"] = pd.cut(
    results_m4["actual"],
    bins=[0, 3, 10, 20, 40, 100],
    labels=["1-3 laps", "4-10 laps", "11-20 laps", "21-40 laps", "41+ laps"],
)
bucket_mae = (
    results_m4.groupby("bucket", observed=False)
    .apply(lambda x: mean_absolute_error(x["actual"], x["pred"]), include_groups=False)
)
print("── M4 MAE by bucket ────────────────────────────────────")
print(bucket_mae)

within_3 = (np.abs(results_m4["actual"] - results_m4["pred"]) <= 3).mean()
within_5 = (np.abs(results_m4["actual"] - results_m4["pred"]) <= 5).mean()
print(f"\nWithin ±3 laps : {within_3:.2%}")
print(f"Within ±5 laps : {within_5:.2%}")


---
## Model 5 · Short-Horizon Laps-Until-Pit Regressor (CatBoost + Optuna)

**Target:** `LapsUntilNextPit` — filtered to **≤ 10 laps** (imminent pit window)  
**Purpose:** Higher-fidelity prediction when the pit window is close  
**Algorithm:** CatBoostRegressor · Optuna TPE · 50 trials · minimise MAE  
**Sample weights:** 3× for laps ≤ 3 (critical window)  
**Achieved:** OOF RMSE **2.898** · Test RMSE **2.786** · MAE **2.292**


In [ ]:
# ── 5-a  Dataset (subset of M4 with LapsUntilNextPit ≤ 10) ───────────────────
df_m5 = df_m4[df_m4["LapsUntilNextPit"] <= 10].copy()
regression_target_eda(df_m5, "LapsUntilNextPit")

X_5 = df_m5.drop(columns=["LapsUntilNextPit"])
y_5 = df_m5["LapsUntilNextPit"]

groups_m5    = df_m5["Year"].astype(str) + "_" + df_m5["EventName"]
splits_m5    = time_based_split(df_m5, X_5, y_5, groups_m5)
X_train      = splits_m5["X_train"];  y_train      = splits_m5["y_train"]
X_valid      = splits_m5["X_valid"];  y_valid      = splits_m5["y_valid"]
X_test       = splits_m5["X_test"];   y_test       = splits_m5["y_test"]
groups_train = splits_m5["groups_train"]

print(f"Short-horizon set — train: {X_train.shape}  test: {X_test.shape}")
# Reuses FeatureEngineeringM4 and preprocessing_m4 unchanged


In [ ]:
# ── 5-b  Optuna hyperparameter search ─────────────────────────────────────────
def objective_m5(trial):
    params = {
        "loss_function": "RMSE", "eval_metric": "RMSE",
        "iterations": 1500,
        "depth":               trial.suggest_int(  "depth",                4,  10),
        "learning_rate":       trial.suggest_float( "learning_rate",      0.01, 0.15, log=True),
        "l2_leaf_reg":         trial.suggest_float( "l2_leaf_reg",         1.0, 30.0, log=True),
        "random_strength":     trial.suggest_float( "random_strength",     0.0, 10.0),
        "bagging_temperature": trial.suggest_float( "bagging_temperature", 0.0, 10.0),
        "min_data_in_leaf":    trial.suggest_int(  "min_data_in_leaf",     5,  100),
        "grow_policy":         trial.suggest_categorical("grow_policy",
                                                          ["SymmetricTree", "Depthwise"]),
        "random_seed": 42, "verbose": 0,
    }
    pipe = Pipeline([
        ("fe",    FeatureEngineeringM4()),
        ("pre",   preprocessing_m4),
        ("model", CatBoostRegressor(**params)),
    ])
    pipe.fit(X_train, y_train)
    return mean_absolute_error(y_valid, pipe.predict(X_valid))

study_m5 = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
study_m5.optimize(objective_m5, n_trials=50, show_progress_bar=True)

best_params_m5 = study_m5.best_params
print(f"\nBest Validation MAE : {study_m5.best_value:.4f}")
print(f"Best Params:\n{best_params_m5}")


In [ ]:
# ── 5-c  5-fold OOF with sample weights (3× for ≤ 3-lap window) ───────────────
oof_preds_m5 = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups_train)):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_vl, y_vl = X_train.iloc[val_idx], y_train.iloc[val_idx]

    weights = np.where(y_tr <= 3, 3.0, 1.0)   # up-weight critical pit window

    fold_pipe = Pipeline([
        ("fe",    FeatureEngineeringM4()),
        ("pre",   preprocessing_m4),
        ("model", CatBoostRegressor(
            iterations=2500, loss_function="RMSE", eval_metric="RMSE",
            **best_params_m5
        )),
    ])
    fold_pipe.fit(X_tr, y_tr, model__sample_weight=weights)
    preds_fold            = fold_pipe.predict(X_vl)
    oof_preds_m5[val_idx] = preds_fold

    print(f"  Fold {fold+1} | MAE={mean_absolute_error(y_vl, preds_fold):.4f} "
          f"| RMSE={root_mean_squared_error(y_vl, preds_fold):.4f}")

print(f"\n── M5 OOF ──────────────────────────────────────────────")
print(f"  OOF MAE  : {mean_absolute_error(y_train, oof_preds_m5):.4f}")
print(f"  OOF RMSE : {root_mean_squared_error(y_train, oof_preds_m5):.4f}")


In [ ]:
# ── 5-d  Final model — evaluate on test ──────────────────────────────────────
m5_pipeline = Pipeline([
    ("fe",    FeatureEngineeringM4()),
    ("pre",   preprocessing_m4),
    ("model", CatBoostRegressor(
        iterations=2500, loss_function="RMSE", eval_metric="RMSE",
        **best_params_m5
    )),
])
m5_pipeline.fit(X_train, y_train)
preds_m5 = m5_pipeline.predict(X_test)

print("── M5 Test Scores ─────────────────────────────────────")
print(f"  MAE  : {mean_absolute_error(y_test, preds_m5):.4f}")
print(f"  RMSE : {root_mean_squared_error(y_test, preds_m5):.4f}")


In [ ]:
# ── 5-e  OOF error analysis by bucket & directional bias ─────────────────────
results_m5 = pd.DataFrame({"actual": y_train, "pred": oof_preds_m5})
results_m5["error"] = results_m5["pred"] - results_m5["actual"]

print("── M5 OOF MAE + bias by bucket ─────────────────────────")
for lo, hi in [(1, 3), (4, 10)]:
    sub = results_m5[results_m5["actual"].between(lo, hi)]
    print(f"  {lo}-{hi} laps | N={len(sub):5d} "
          f"| MAE={mean_absolute_error(sub['actual'], sub['pred']):.3f} "
          f"| Bias={sub['error'].mean():.3f}")


---
## 6 · Results Summary

| Model | Target | Algorithm | Test RMSE | Test MAE | ROC-AUC | PR-AUC |
|-------|--------|-----------|:---------:|:--------:|:-------:|:------:|
| M1 | LapTimeSec | LightGBM + Optuna | 4.298 | 1.971 | — | — |
| M2 | IsPitLap | CatBoost + Optuna | — | — | 0.8226 | 0.1439 |
| M3 | WillPitIn3Laps | CatBoost + Optuna | — | — | 0.7856 | 0.2606 |
| M4 | LapsUntilNextPit (all) | CatBoost + Optuna | 7.677 | 5.503 | — | — |
| M5 | LapsUntilNextPit (≤10) | CatBoost + Optuna | 2.786 | 2.292 | — | — |

### Key design decisions
- **Temporal split** — no future-season data ever enters the training set.
- **GroupKFold on `Year_EventName`** — lap sequences from the same race are never split across folds.
- **OOF stacking** — M2 pit-probabilities generated out-of-fold so M3/M4 never train on in-sample leakage.
- **Sample weighting in M5** — 3× weight on ≤3-lap rows to sharpen the critical pit-window prediction.
- **log1p target in M4** — reduces sensitivity to the long right tail of `LapsUntilNextPit`.
